# Problem Set 4 – My First Quantum Algorithm: A "Quantum Stock Market Identifier"

**Name:** _your name here_  
**Section:** _A / B_  
**Date submitted:** _YYYY-MM-DD_

**Submitting:** when you're done, *Restart & Run All*, save (`Ctrl+S`), then right-click this file in the Explorer → **Download…** and upload the `.ipynb` to Canvas.

## ⚠️ Old Qiskit code you will find online (and from AI)

Qiskit 1.0 and 2.0 removed several things. AI tools and older tutorials still write them. **If you see any of these, the code is outdated and will fail:**

| Outdated (will error) | Use this instead (Qiskit 2.x) |
|---|---|
| `from qiskit import execute` / `execute(qc, backend)` | `tqc = transpile(qc, sim)` then `sim.run(tqc, shots=1024).result()` |
| `from qiskit import Aer` / `Aer.get_backend('qasm_simulator')` | `from qiskit_aer import AerSimulator` then `sim = AerSimulator()` |
| `from qiskit import IBMQ` / `IBMQ.load_account()` | `from qiskit_ibm_runtime import QiskitRuntimeService` (only for real hardware) |
| `qiskit.providers.ibmq` | `qiskit_ibm_runtime` |
| `BasicAer`, `qiskit.test`, `qiskit.opflow` | removed – use `qiskit_aer`, `qiskit.quantum_info` |
| `Sampler()` / `Estimator()` from `qiskit.primitives` | `StatevectorSampler()` / `StatevectorEstimator()` (or `qiskit_aer.primitives.SamplerV2`) |

Two current patterns that always work here:

```python
# Pattern A: simulator backend
from qiskit import transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()

# Pattern B: primitives (exact statevector sampling, no Aer needed)
from qiskit.primitives import StatevectorSampler
result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.c.get_counts()   # 'c' = name of your classical register
```

**Bit order:** Qiskit prints bitstrings with qubit 0 on the **right**. `'01'` means q0 = 1, q1 = 0.

## Setup (run first)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

sim = AerSimulator()

## Data Generation: Five Noisy Stock Oracles

Generate 256 days of stock data for five patterns with 26 flipped days each.

In [ ]:
# Five clean patterns (before noise)
def pattern_1(x):
    """Constant up: f(x) = 1 for all x"""
    return 1

def pattern_2(x):
    """Constant down: f(x) = 0 for all x"""
    return 0

def pattern_3(x):
    """Alternating (parity): f(x) = x mod 2 = x_0"""
    return x % 2

def pattern_4(x):
    """Bull market: first half up (x <= 127), second half down (x >= 128)"""
    return 1 if x <= 127 else 0

def pattern_5(x):
    """Cyclical: quarters up, down, up, down (up for x in [0,63] U [128,191])"""
    return 1 if (0 <= x <= 63) or (128 <= x <= 191) else 0

patterns = [pattern_1, pattern_2, pattern_3, pattern_4, pattern_5]

# Generate noisy data for all five patterns
noisy_data = {}
for p in range(1, 6):
    # Start with clean pattern
    data = [patterns[p-1](x) for x in range(256)]
    
    # Add noise: flip 26 days chosen with fixed seed
    flip_days = np.random.default_rng(2026 + p).choice(256, size=26, replace=False)
    for day in flip_days:
        data[day] = 1 - data[day]  # flip the bit
    
    noisy_data[p] = data

print(f"Generated noisy data for {len(noisy_data)} patterns, each with 256 days.")
print(f"Example: Pattern 1 noisy data (first 10 days): {noisy_data[1][:10]}")

---
# Question 1: Lookup-table oracle

Build $U_f$ (8 input qubits + 1 ancilla) from the 256-entry noisy data list. For each day $d$ with $f(d)=1$, add one 8-control MCX gate that fires only on input $|d\rangle$.

### 1a. Code: Build all five noisy oracles and check one on a few input days

In [ ]:
# Build lookup-table oracle from data list
# Your code here


### 1b. Explanation (your own words, cite your own gate counts/outputs)

Make a table with each oracle's `count_ops()` before transpiling and after transpiling to `cx` and `u` gates (`optimization_level=1`): number of `mcx`, `cx`, and depth. Which oracle is largest, and why that one? Explain why building this oracle required reading every one of the 256 days. Once it's built, what would DJ tell you that you didn't already know?

---
# Question 2: Formula oracle

The clean patterns follow simple rules.

### 2a. Code: Pick one noiseless pattern (3, 4, or 5), write f(x) as a rule, build U_f with at most two gates

In [ ]:
# Build formula-based oracle for one clean pattern
# Your code here


### 2a. Explanation (your own words, cite your own gate counts/outputs)

Show it matches the noiseless lookup-table version (same unitary, or same DJ output).

### 2b. Explanation (your own words, cite your own gate counts/outputs)

Compare its gate count to the lookup version from your own runs and explain the difference. What information did each construction need? Now the noisy version: can it be written as a short rule? What is the cheapest oracle you can build for it (hint: rule + exceptions)? Count its gates. What did you need to know to build it?

---
# Question 3: Run DJ and read the output

Run Deutsch-Jozsa on all five noisy oracles.

### 3a. Code: Run DJ on all five noisy oracles (2000 shots verification run)

In [ ]:
# Deutsch-Jozsa algorithm implementation
# Your code here


### 3b. Explanation (your own words, cite your own gate counts/outputs)

For patterns 3–5, the most frequent bitstring is not `00000000`. In each case, which qubit is 1 (Qiskit prints $q_7$ on the left)? How does that match the bit in your rule from Question 2? Constant up and constant down give the same result. Explain why, using the phrase *global phase*, and say what one extra classical query of $f$ adds.

### 3c. Explanation (your own words, cite your own gate counts/outputs)

The expected bitstring now appears only about 64% of the time. Compare your measured frequencies to $\left(\frac{256-2k}{256}\right)^2$ with $k=26$. Explain why each flipped day costs $2/256$ of amplitude. *Hint:* the amplitude of output $z$ is $\frac{1}{256}\sum_x(-1)^{f(x)}(-1)^{z\cdot x}$, so what does a flip do to one term? Where does the rest of the probability go?

---
# Question 4: Classify with as few shots as possible

Run each noisy oracle with a small number of shots $s$. Use a classical rule, plus any extra classical queries, to name its pattern.

### 4a. Code: Classification with minimal shots

In [ ]:
# Classify oracles with minimal shots
# Your code here


### 4a. Explanation (your own words, cite your own gate counts/outputs)

Show a table of your classifications and the **total** number of shots and classical queries.

### 4b. Explanation (your own words, cite your own gate counts/outputs)

Justify $s$ using your measured $P\approx0.64$. What is the chance one oracle is misclassified, and the chance at least one of the five is? Check this empirically by repeating your classification many times.

---
# Question 5: So what's the point?

Where does a quantum advantage actually come from?

### 5a. Explanation (your own words, cite your own gate counts/outputs)

Using your Question 1 and 2 numbers, explain where a real speedup *could* come from. One case: the oracle is a black box someone else provides, and you count queries. Another: $f$ is computed by a circuit or program that is cheap to run, but its global property is unknown. Contrast both with the stock data, where the "function" is just stored data. Also mention that without noise, checking about 11 random days classically tells constant from balanced with high confidence.

### 5b. Reflection (3–4 sentences)

What does this leave as DJ's practical value? Why does it still matter historically, and for Bernstein–Vazirani, Simon, and Shor?

---
# Question 6 (Optional): Pick one

Choose either 6a or 6b.

### 6a. Build a noisy oracle with fewer CX gates than your rule-plus-exceptions version

In [ ]:
# Optional 6a code
# Your code here


### 6a. Explanation (your own words, cite your own gate counts/outputs)

Explain where the savings come from.

### 6b. Bernstein–Vazirani: build f(x) = s · x mod 2 for a secret s

In [ ]:
# Optional 6b code
# Your code here


### 6b. Explanation (your own words, cite your own gate counts/outputs)

Show that one shot recovers $s$, and explain why.

---
# AI Use Reflection

- Which part of this problem set did you understand best, and which least?
- **AI use:** which tool(s), for which parts? What did you have to fix or change in the AI's code?